# 03-6. 推定 — 動かして確かめる

📖 解説: [`../06_estimation.md`](../06_estimation.md)

上から順に **Shift+Enter** で実行していってください。

## このノートで触るもの
1. 【対話】標本平均はどれくらいブレるか — $\sqrt{n}$ の法則
2. ⚠️ 信頼区間の本当の意味 — 100 回調査して何回当たるか
3. $t$ 分布 vs 正規分布 — なぜ小標本では $t$ なのか
4. 最尤推定 (MLE) — 尤度関数を描いて最大点を探す
5. JAX で MLE — `grad` で手計算なしに解く
6. 【対話】MAP 推定 — 事前分布の強さを変える
7. 正則化 = MAP という対応の確認

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (06_estimation.md)](../06_estimation.md)

In [ ]:
import numpy as np
from scipy import stats, optimize
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
import jax
import jax.numpy as jnp
from ipywidgets import interact, IntSlider, FloatSlider

%matplotlib inline

rng = np.random.default_rng(42)

## 1. 【対話】標本平均はどれくらいブレるか

$$
\mathrm{SE}(\bar{X}) = \frac{\sigma}{\sqrt{n}}
$$

**注目点**: $n$ を 4 倍にしても、ばらつきは **2 倍しか**小さくなりません。
精度は $\sqrt{n}$ でしか上がらない — これが調査設計の根本的な制約です。

In [ ]:
TRUE_MU: float = 50.0     # 母平均 μ (単位: 点)
TRUE_SIGMA: float = 10.0  # 母標準偏差 σ


def sampling_distribution(n: int = 10) -> None:
    """標本サイズ n のとき、標本平均がどう散らばるかを描く.

    Args:
        n: 1 回の調査で集めるデータ数
    """
    n_experiments: int = 5000
    local = np.random.default_rng(0)
    samples = local.normal(TRUE_MU, TRUE_SIGMA, size=(n_experiments, n))  # shape: (5000, n)
    x_bars = samples.mean(axis=1)                                          # shape: (5000,)

    se_theory = TRUE_SIGMA / np.sqrt(n)

    plt.figure(figsize=(9, 4))
    plt.hist(x_bars, bins=50, density=True, alpha=0.75, label=f'標本平均の分布 (n={n})')
    plt.axvline(TRUE_MU, c='r', ls='--', lw=2, label=f'母平均 μ={TRUE_MU}')
    plt.xlim(TRUE_MU - 4 * TRUE_SIGMA, TRUE_MU + 4 * TRUE_SIGMA)
    plt.xlabel('標本平均 x̄'); plt.ylabel('密度')
    plt.title(f'n={n}:  実測SE {x_bars.std():.3f}   理論SE σ/√n = {se_theory:.3f}')
    plt.legend(); plt.grid(alpha=0.3); plt.show()

    print(f'n = {n:>4}   実測 SE = {x_bars.std():.4f}   理論 SE = {se_theory:.4f}')


interact(sampling_distribution, n=IntSlider(min=1, max=200, step=1, value=10))

In [ ]:
# √n の法則を数値で確認: n を 4 倍にすると SE は半分
print(f'{"n":>8}{"理論 SE":>12}{"1つ前との比":>14}')
print('-' * 34)
prev = None
for n in (10, 40, 160, 640, 2560):
    se = TRUE_SIGMA / np.sqrt(n)
    ratio = f'{se/prev:.3f}' if prev else '—'
    print(f'{n:>8}{se:>12.4f}{ratio:>14}')
    prev = se
print()
print('n を 4 倍にするたび、SE は約 0.5 倍 (= 1/√4)。')
print('データを 4 倍集めても、精度は 2 倍にしかならない。')

## 2. ⚠️ 信頼区間の本当の意味

> ❌ 誤り: 「真の $\mu$ がこの区間に入る確率が 95%」
> ✅ 正しい: 「同じ手順を繰り返せば、**約 95% の区間**が真の $\mu$ を含む」

言葉だけではピンと来ないので、**実際に 100 回調査して数えます**。

In [ ]:
def ci_simulation(n: int = 20, n_studies: int = 100) -> None:
    """100 回調査を繰り返し、信頼区間が母平均を含む割合を数える.

    Args:
        n: 1 回の調査のサンプルサイズ
        n_studies: 調査を繰り返す回数
    """
    local = np.random.default_rng(7)
    hits = 0
    fig, ax = plt.subplots(figsize=(10, 5))

    for i in range(n_studies):
        sample = local.normal(TRUE_MU, TRUE_SIGMA, n)          # shape: (n,)
        mean = sample.mean()
        se = sample.std(ddof=1) / np.sqrt(n)                   # 標準誤差
        t_crit = stats.t.ppf(0.975, df=n - 1)                  # 自由度 n-1
        lo, hi = mean - t_crit * se, mean + t_crit * se

        covered = lo <= TRUE_MU <= hi
        hits += covered
        ax.plot([lo, hi], [i, i], c='tab:blue' if covered else 'tab:red',
                lw=1.2, alpha=0.8)

    ax.axvline(TRUE_MU, c='k', ls='--', lw=2, label=f'母平均 μ={TRUE_MU}')
    ax.set_xlabel('95% 信頼区間'); ax.set_ylabel('調査の回数')
    ax.set_title(f'{n_studies} 回の調査: {hits} 回が μ を含んだ ({hits/n_studies:.0%})  '
                 f'赤 = 外した区間')
    ax.legend(); plt.tight_layout(); plt.show()

    print(f'μ を含んだ区間: {hits}/{n_studies} = {hits/n_studies:.1%}   (理論値 95%)')
    print()
    print('★ 赤い線 (外した区間) も必ず出ます。それが「5%」の意味。')
    print('  区間そのものが調査ごとに動いていることに注目してください。')
    print('  動かないのは μ (黒い破線) のほう。だから「μ が入る確率」とは言えない。')


ci_simulation()

## 3. $t$ 分布 vs 正規分布 — なぜ小標本では $t$ なのか

母分散 $\sigma^2$ が未知なので $s^2$ で代用します。
そのぶん**不確実性が二重**になるので、正規分布より**裾が重い** $t$ 分布を使います。

In [ ]:
x = np.linspace(-5, 5, 400)

plt.figure(figsize=(9, 4.5))
plt.plot(x, stats.norm.pdf(x), 'k-', lw=2.5, label='正規分布 N(0,1)')
for df in (1, 3, 10, 30):
    plt.plot(x, stats.t.pdf(x, df), lw=1.5, ls='--', label=f't 分布 (自由度 {df})')
plt.xlabel('x'); plt.ylabel('密度')
plt.title('自由度が上がるほど t 分布は正規分布に近づく')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print(f'{"n":>6}{"自由度":>8}{"t 臨界値":>12}{"z 臨界値":>12}{"差":>10}')
print('-' * 48)
z_crit = stats.norm.ppf(0.975)
for n in (5, 10, 30, 100, 1000):
    t_crit = stats.t.ppf(0.975, n - 1)
    print(f'{n:>6}{n-1:>8}{t_crit:>12.3f}{z_crit:>12.3f}{t_crit - z_crit:>10.3f}')
print()
print('n が小さいほど t は大きい = 区間が広い = 「自信がない」ことを正しく表現している。')
print('n ≥ 30 くらいから差はごくわずか。だから「n≥30 なら正規近似でよい」と言われる。')

## 4. 最尤推定 (MLE) — 尤度関数を描いてみる

> 「このデータが観測される確率が最も高くなるパラメータ」を選ぶ。

コインを 10 回投げて 7 回表が出たとき、$p$ はいくつが最もありそうか。

In [ ]:
n_trials: int = 10
n_heads: int = 7

p_grid = np.linspace(0.01, 0.99, 300)                                  # shape: (300,)
log_lik = n_heads * np.log(p_grid) + (n_trials - n_heads) * np.log(1 - p_grid)

plt.figure(figsize=(9, 4))
plt.plot(p_grid, log_lik, lw=2)
plt.axvline(n_heads / n_trials, c='r', ls='--', lw=2,
            label=f'最大点 p̂ = {n_heads/n_trials:.2f}')
plt.xlabel('p (表が出る確率)'); plt.ylabel('対数尤度 ℓ(p)')
plt.title('10 回中 7 回表だったときの対数尤度')
plt.legend(); plt.grid(alpha=0.3); plt.show()


def neg_log_likelihood(p: float) -> float:
    """二項分布の負の対数尤度.

    Args:
        p: 表が出る確率 (0 < p < 1)

    Returns:
        負の対数尤度
    """
    return -(n_heads * np.log(p) + (n_trials - n_heads) * np.log(1 - p))


result = optimize.minimize_scalar(neg_log_likelihood, bounds=(1e-6, 1 - 1e-6), method='bounded')
print(f'数値解 : p̂ = {result.x:.6f}')
print(f'解析解 : p̂ = k/n = {n_heads / n_trials:.6f}')
print('→ 一致。最尤推定は「素朴な直感」を数学的に正当化してくれる。')

## 5. JAX で MLE — `grad` で手計算なしに解く

最尤推定は「対数尤度を微分してゼロにする」問題なので、**自動微分と相性が最高**です。
正規分布の $\mu$ と $\sigma$ を、同時に推定してみます。

In [ ]:
key = jax.random.PRNGKey(0)
data: jnp.ndarray = 3.0 + 2.0 * jax.random.normal(key, shape=(500,))  # 真値 μ=3, σ=2


def neg_log_lik(params: jnp.ndarray, x: jnp.ndarray) -> jnp.ndarray:
    """正規分布の負の対数尤度 (平均版).

    Args:
        params: [μ, log σ]  σ>0 を保証するため log で持つ, shape: (2,)
        x: 観測データ, shape: (N,)

    Returns:
        負の対数尤度 (スカラー)
    """
    mu, log_sigma = params[0], params[1]
    sigma = jnp.exp(log_sigma)
    # sum ではなく mean。sum だと勾配が N 倍になり発散しやすい
    return jnp.mean(log_sigma + 0.5 * ((x - mu) / sigma) ** 2)


grad_fn = jax.grad(neg_log_lik)      # ← 手で微分しない
params = jnp.array([0.0, 0.0])       # 初期値 μ=0, log σ=0

history = []
for step in range(2000):
    params = params - 0.05 * grad_fn(params, data)
    if step % 20 == 0:
        history.append([float(params[0]), float(jnp.exp(params[1]))])

hist = np.array(history)             # shape: (100, 2)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(hist[:, 0]); axes[0].axhline(3.0, c='r', ls='--', label='真値 3.0')
axes[0].set_title('μ の収束'); axes[0].set_xlabel('step (×20)'); axes[0].legend()
axes[1].plot(hist[:, 1]); axes[1].axhline(2.0, c='r', ls='--', label='真値 2.0')
axes[1].set_title('σ の収束'); axes[1].set_xlabel('step (×20)'); axes[1].legend()
plt.tight_layout(); plt.show()

print(f'推定 μ̂ = {params[0]:.4f}   (真値 3.0)')
print(f'推定 σ̂ = {jnp.exp(params[1]):.4f}   (真値 2.0)')
print()
print('検算 — 正規分布の MLE は解析的に「標本平均」「標本SD(ddof=0)」になる:')
print(f'  標本平均        = {jnp.mean(data):.4f}')
print(f'  標本SD (ddof=0) = {jnp.std(data):.4f}')
print('→ 勾配降下の結果と一致。理論通り。')

## 6. 【対話】MAP 推定 — 事前分布を足す

最尤推定の弱点: **データが少ないと極端な答えを出す**。
2 回投げて 2 回表なら $\hat{p}_{\text{MLE}} = 1.0$（「絶対に裏が出ない」）。

事前分布 $\mathrm{Beta}(\alpha,\beta)$ を置いた MAP 推定はこうなります:

$$
\hat{p}_{\text{MAP}} = \frac{k + \alpha - 1}{n + \alpha + \beta - 2}
$$

**注目点**: データ数 $n$ を増やすと、事前分布の影響は薄れて MAP は MLE に近づきます。

In [ ]:
def map_vs_mle(n_flips: int = 2, prior_strength: float = 2.0) -> None:
    """MLE と MAP を、事前分布の強さを変えながら比較する.

    Args:
        n_flips: コインを投げる回数
        prior_strength: 事前分布 Beta(a,a) の a。大きいほど「半々」という信念が強い
    """
    true_p: float = 0.5
    local = np.random.default_rng(3)
    k = int(local.binomial(n_flips, true_p))     # 表の回数

    a = b = prior_strength
    mle = k / n_flips
    map_est = (k + a - 1) / (n_flips + a + b - 2)

    # 事前分布・尤度・事後分布を描く
    p = np.linspace(0.001, 0.999, 400)
    prior = stats.beta.pdf(p, a, b)
    likelihood = p**k * (1 - p)**(n_flips - k)
    likelihood = likelihood / np.trapezoid(likelihood, p)         # 面積 1 に正規化
    posterior = stats.beta.pdf(p, a + k, b + n_flips - k)

    plt.figure(figsize=(9, 4))
    plt.plot(p, prior, ls=':', lw=2, label=f'事前分布 Beta({a:.0f},{b:.0f})')
    plt.plot(p, likelihood, ls='--', lw=2, label='尤度 (データだけ)')
    plt.plot(p, posterior, lw=2.5, label='事後分布')
    plt.axvline(mle, c='tab:orange', ls='--', label=f'MLE = {mle:.3f}')
    plt.axvline(map_est, c='tab:green', lw=2, label=f'MAP = {map_est:.3f}')
    plt.axvline(true_p, c='k', ls='-', lw=1, alpha=0.5, label=f'真値 {true_p}')
    plt.xlabel('p'); plt.ylabel('密度')
    plt.title(f'{n_flips} 回中 {k} 回表:  MLE={mle:.3f} / MAP={map_est:.3f}')
    plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

    print(f'投げた回数 {n_flips:>4} / 表 {k:>4} 回')
    print(f'  MLE = {mle:.4f}   ← データだけ')
    print(f'  MAP = {map_est:.4f}   ← 「半々のはず」という事前知識を加味')


interact(map_vs_mle,
         n_flips=IntSlider(min=2, max=200, step=2, value=2),
         prior_strength=FloatSlider(min=1.0, max=20.0, step=1.0, value=2.0))

## 7. 正則化 = MAP 推定という対応

解説の第 6 節で見たとおり、**L2 正則化はガウス事前分布を置いた MAP 推定**と同じでした。

$$
\underbrace{\min_w \sum (y_i - w x_i)^2 + \lambda w^2}_{\text{Ridge 回帰}}
\quad\Longleftrightarrow\quad
\underbrace{\max_w \log P(D \mid w) + \log \mathcal{N}(w \mid 0, \tau^2)}_{\text{MAP 推定}}
$$

数値で確認します。$\lambda$ を大きくする = 事前分布を狭くする = 係数が 0 に引き寄せられる。

In [ ]:
# 単純な線形モデル y = w x + ノイズ で、Ridge の λ と MAP の事前分散の対応を見る
reg_rng = np.random.default_rng(1)
n_data = 20
TRUE_W = 2.0
x_d = reg_rng.uniform(-2, 2, n_data)                      # shape: (20,)
y_d = TRUE_W * x_d + reg_rng.normal(0, 1.0, n_data)       # shape: (20,)

print(f'{"λ":>8}{"Ridge の ŵ":>14}{"対応する事前SD τ":>18}')
print('-' * 42)
for lam in (0.0, 1.0, 10.0, 100.0):
    # Ridge の解析解: w = Σxy / (Σx² + λ)
    w_ridge = (x_d @ y_d) / (x_d @ x_d + lam)
    # λ = σ²/τ² の関係 (σ=1 とした)  → τ = 1/√λ
    tau = np.inf if lam == 0 else 1.0 / np.sqrt(lam)
    tau_s = '∞ (無情報)' if lam == 0 else f'{tau:.3f}'
    print(f'{lam:>8.0f}{w_ridge:>14.4f}{tau_s:>18}')

print()
print(f'真の w = {TRUE_W}')
print('λ が大きい = 事前分布が狭い = 「w は 0 に近いはず」という信念が強い')
print('→ 推定値が 0 に引き寄せられる (縮小推定)')

## まとめ

- **標準誤差** $\sigma/\sqrt{n}$。データを 4 倍にしても精度は 2 倍にしかならない
- **信頼区間**は「手順を繰り返せば 95% が当たる」。動くのは区間であって $\mu$ ではない
- **$t$ 分布**は $\sigma$ を $s$ で代用した不確実性を吸収する。$n$ が大きければ $z$ に近づく
- **MLE** は「観測データが最も起こりやすい $\theta$」。`jax.grad` で手計算なしに解ける
- **MAP** = MLE + 事前分布。データが増えれば MLE に近づく
- **L2 正則化はガウス事前分布の MAP 推定**だった

次は、推定した差が「本物か偶然か」を判定する話へ。

→ 次: [`07_hypothesis_testing.ipynb`](07_hypothesis_testing.ipynb)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`05_descriptive_stats.ipynb`](05_descriptive_stats.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`07_hypothesis_testing.ipynb`](07_hypothesis_testing.ipynb) |